This is the jupyter notebook to derive ACP and RESEDA scenario-specific temporally differentiated marginal characterization factors of dissipative of metallic elements and fossil resources (crude oil, natural gas, coal)

More info on these two life cycle impact assessment methods at: https://doi.org/10.1007/s11367-026-02593-5

In [30]:
import pandas as pd

In [3]:
def BtH_value(df, e_i, e_j):
    try:
        if e_i == e_j:
            return 1
        else:
            return df.loc[e_i, e_j]
    except:
        return (0)
def always_value_capacity(df, m, t):
    try:
        return (df[(df.loc[:, "constraint"] == "c2_prod") & (df.loc[:, "Year"] == t)].loc[m, "Amount (tons/year)"])
    except:
        pass
    try:
        if t> 2050:
            a = df[(df.loc[:, "constraint"] == "c2_prod") & (df.loc[:, "Year"] == 2050)].loc[m, "Amount (tons/year)"]
            return(df[(df.loc[:, "constraint"] == "c2_prod") & (df.loc[:, "Year"] == 2050)].loc[m, "Amount (tons/year)"])
    except:
        pass
    try:
        return (
            df[(df.loc[:, "constraint"] == "c2_prod") & (df.loc[:, "Year"] == "All")].loc[m, "Amount (tons/year)"])
    except:
        return (1E11)
def set_cf_reseda_1(df):
    for i in df.index:
        for j in df.columns:
            if df.loc[i,j] >0:
                df.loc[i,j] = 1
    return df

def format_e(n):
    a = '%E' % n
    return a.split('E')[0].rstrip('0').rstrip('.') + 'E' + a.split('E')[1]

ts = 1
def mining_cost_calc(elem,f_ext,r0,r1,r2,r3,f0,f1,f2,f3,mining_cost,gap_reserves,ts):
    for e in elements:
        for t in time[1:]:
            if (r0.loc[e,t-ts] >= f_ext.loc[e,t]):
                f0.loc[e, t] = f_ext.loc[e,t]
                r0.loc[e,t] = r0.loc[e,t-ts]-f0.loc[e, t]
                r1.loc[e,t] = r1.loc[e,t-ts]
                r2.loc[e,t] = r2.loc[e,t-ts]
                r3.loc[e,t] = r3.loc[e,t-ts]
            if (f_ext.loc[e,t] >= r0.loc[e,t-ts]) and (
                f_ext.loc[e,t] < (r0.loc[e,t-ts]+r1.loc[e,t-ts])):
                f0.loc[e, t] = r0.loc[e,t-ts]
                f1.loc[e, t] = f_ext.loc[e,t]-f0.loc[e, t]
                r0.loc[e,t] = r0.loc[e,t-ts]-f0.loc[e, t]
                r1.loc[e,t] = r1.loc[e,t-ts]-f1.loc[e, t]
                r2.loc[e,t] = r2.loc[e,t-ts]
                r3.loc[e,t] = r2.loc[e,t-ts]
            if (f_ext.loc[e,t] >= (r0.loc[e,t-ts]+r1.loc[e,t-ts])) and (
                f_ext.loc[e,t] < (r0.loc[e,t-ts]+r1.loc[e,t-ts]+r2.loc[e,t-ts])):
                f0.loc[e, t] = r0.loc[e,t-ts]
                f1.loc[e, t] = r1.loc[e,t-ts]
                f2.loc[e, t] = f_ext.loc[e,t]-(f0.loc[e, t]+f1.loc[e, t])
                r0.loc[e,t] = r0.loc[e,t-ts]-f0.loc[e, t]
                r1.loc[e,t] = r1.loc[e,t-ts]-f1.loc[e, t]
                r2.loc[e,t] = r2.loc[e,t-ts]-f2.loc[e, t]
                r3.loc[e,t] = r3.loc[e,t-ts]
            if (f_ext.loc[e,t] >= (r0.loc[e,t-ts]+r1.loc[e,t-ts]+r2.loc[e,t-ts])) and (
                f_ext.loc[e,t] < (r0.loc[e,t-ts]+r1.loc[e,t-ts]+r2.loc[e,t-ts]+r3.loc[e,t-ts])):
                f0.loc[e, t] = r0.loc[e,t-ts]
                f1.loc[e, t] = r1.loc[e,t-ts]
                f2.loc[e, t] = r2.loc[e,t-ts]
                f3.loc[e, t] = f_ext.loc[e,t]-(f0.loc[e, t]+f1.loc[e, t]+f2.loc[e, t])
                r0.loc[e,t] = r0.loc[e,t-ts]-f0.loc[e, t]
                r1.loc[e,t] = r1.loc[e,t-ts]-f1.loc[e, t]
                r2.loc[e,t] = r2.loc[e,t-ts]-f2.loc[e, t]
                r3.loc[e,t] = r3.loc[e,t-ts]-f3.loc[e, t]
            if (f_ext.loc[e,t] > (r0.loc[e,t-ts]+r1.loc[e,t-ts]+r2.loc[e,t-ts]+r3.loc[e,t-ts])):
                f0.loc[e, t] = r0.loc[e,t-ts]
                f1.loc[e, t] = r1.loc[e,t-ts]
                f2.loc[e, t] = r2.loc[e,t-ts]
                f3.loc[e, t] = r3.loc[e,t-ts]
                r0.loc[e,t] = r0.loc[e,t-ts]-f0.loc[e, t]
                r1.loc[e,t] = r1.loc[e,t-ts]-f1.loc[e, t]
                r2.loc[e,t] = r2.loc[e,t-ts]-f2.loc[e, t]
                r3.loc[e,t] = r3.loc[e,t-ts]-f3.loc[e, t]
                gap_reserves.loc[e,t] = f_ext.loc[e,t]-(f0.loc[e, t]+f1.loc[e, t]+f2.loc[e, t]+f3.loc[e, t])
            max_ = max(costs.loc[e,"C_Level_0"],costs.loc[e,"C_Level_1"],
                      costs.loc[e,"C_Level_2"],costs.loc[e,"C_Level_3"])
            
        
            mining_cost.loc[e,t] = (costs.loc[e,"C_Level_0"]*f0.loc[e,t]+
                                   costs.loc[e,"C_Level_1"]*f1.loc[e,t]+
                                   costs.loc[e,"C_Level_2"]*f2.loc[e,t]+
                                   costs.loc[e,"C_Level_3"]*f3.loc[e,t]+
                                   max_*gap_reserves.loc[e,t])
    return (mining_cost,sum(mining_cost.loc[e,t] for t in time),gap_reserves)


Download "SI_1_Inputs_WILMFlo_v1.01.xlsx" and "Results.zip" from https://zenodo.org/records/19208690

Create a folder "Input" on your computer and add in this folder "SI_1_Inputs_WILMFlo_v1.01.xlsx", "Results_DLS_IEA_NZ_.xlsx", "Results_STEPS_IEA_NZ_.xlsx" and "Results_STEPS_STEPS_.xlsx".
Create a folder "Output" in which you can export scenario-specific temporally differentiated marginal characterization factors of dissipative flows of metallic elements and fossil resources

In [15]:
### Change paths of folder_input and folder_output so that it can find files in "Input" folder on your computer

## Advice: Keep the r"/" structure so that pandas can read excel files
folder_input = r"[path to folder]\Input/"
folder_output = r"[path to folder]\Output/"
file_input = "SI_1_Inputs_WILMFlo_v1.01.xlsx"

In [16]:
costs = pd.read_excel(folder_input+file_input, sheet_name = "Reserves_cost", index_col=0)
lists = pd.read_excel(folder_input+file_input, sheet_name = "Lists")
excluded = list(set(lists.loc[:,"No_Tec_Yet"].dropna()))
all_elements = [i for i in list(set(lists.loc[:,"Element"].dropna())) if i not in excluded]
materials = list(set(lists.loc[:,"Material"].dropna()))

In [23]:
folder_scenario = "Results/"
#file = "Results_DLS_IEA_NZ_.xlsx" #### to derive DLS NZ scenario
#file = "Results_STEPS_STEPS_.xlsx" #### to derive STEPS scenario
file = "Results_STEPS_IEA_NZ_.xlsx" #### to derive NZ scenario

scenarios = ["DLS_IEA_NZ","NZ","STEPs"]
scenario = scenarios[1] ### 0 for DLS NZ, 1 for NZ and 2 for STEPs

f_ext = pd.read_excel(folder_input+file, sheet_name = "f_ext", index_col=0)
f_ext_supply = pd.read_excel(folder_input+file, sheet_name = "f_ext_supply", index_col=0)
capacity = pd.read_excel(folder_input+file_input, sheet_name = "Capacity", index_col=0)
mat_BtH = pd.read_excel(folder_input+file_input, sheet_name="av_BtH", index_col=0)
elements = list(f_ext.index)
hosts = [i for i in list(lists.loc[:,"Hosts"].dropna())]
byproducts = [i for i in list(lists.loc[:,"Byproducts"].dropna())]
pure_byproducts = [i for i in byproducts if i not in hosts]
other_elements = [i for i in elements if i not in pure_byproducts]
time = list(range(2024,2101))


## Df for reference scenario
r0_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r1_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r2_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r3_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
gap_reserves_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f0_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f1_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f2_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f3_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)
upper_bound_f_ext = pd.DataFrame(0,index=elements, columns=time,dtype=float)
deficit_extraction_cap = pd.DataFrame(0,index=elements, columns=time,dtype=float)
surplus_extraction_cap = pd.DataFrame(0,index=elements, columns=time,dtype=float)
mining_cost_ref = pd.DataFrame(0,index=elements, columns=time,dtype=float)

## Df for marginal scenario
r0_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r1_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r2_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
r3_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
gap_reserves_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f0_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f1_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f2_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
f3_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
mining_cost_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)

In [24]:
for e in elements:
    for t in time:
        r0_ref.loc[e,2022] = costs.loc[e,"R_Level_0"]
        r1_ref.loc[e,2022] = costs.loc[e,"R_Level_1"]
        r2_ref.loc[e,2022] = costs.loc[e,"R_Level_2"]
        r3_ref.loc[e,2022] = costs.loc[e,"R_Level_3"]

In [25]:
for t in time:
    for e in other_elements:
        f_ext.loc[e, t] = min(always_value_capacity(capacity, e, t), f_ext_supply.loc[e, t])
        deficit_extraction_cap.loc[e, t] = max(0, f_ext_supply.loc[e, t] - always_value_capacity(capacity, e, t))
    for b in pure_byproducts:
        upper_bound_f_ext.loc[b,t] = min(always_value_capacity(capacity, b, t),sum(f_ext.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts))
        f_ext.loc[b, t] = min(upper_bound_f_ext.loc[b,t], f_ext_supply.loc[b, t])
        deficit_extraction_cap.loc[b,t] = max(0,f_ext_supply.loc[b,t]-upper_bound_f_ext.loc[b,t])
        surplus_extraction_cap.loc[b,t] = max(0,sum(f_ext.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts)-f_ext_supply.loc[b, t])

In [26]:
years = list(range(2025,2035))
CF_RESEDA = pd.DataFrame(0,index = elements, columns = years, dtype=float)
CF_RESEDA_average = pd.DataFrame(0, index = elements, columns=  ["kg deficit/kg dissipated"], dtype=float)
CF_ACP = pd.DataFrame(0,index = elements, columns = years, dtype=float)
CF_ACP_average = pd.DataFrame(0, index = elements, columns=  ["MJ/kg dissipated"], dtype=float)

In [28]:
fract, ts = 0.01, 1
selec_years = list(range(2025,2035))
df_cost_ref,total_ref, new_gap_reserves = mining_cost_calc("",f_ext,r0_ref,r1_ref,r2_ref,r3_ref,f0_ref,f1_ref,f2_ref,f3_ref,mining_cost_ref,gap_reserves_ref,ts)

#sel_elements = ["Al","Li","Co","Cu","Te","Nd"]
print("Scenario = ",scenario)
for year in selec_years:
    for element in elements:
        upper_bound_f_ext_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        deficit_extraction_cap_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        surplus_extraction_cap_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        f_ext_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        
        r0_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        r1_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        r2_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        r3_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        gap_reserves_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        f0_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        f1_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        f2_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        f3_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        mining_cost_marg = pd.DataFrame(0,index=elements, columns=time,dtype=float)
        for e in elements:
            for t in time:
                r0_marg.loc[e,2022] = costs.loc[e,"R_Level_0"]
                r1_marg.loc[e,2022] = costs.loc[e,"R_Level_1"]
                r2_marg.loc[e,2022] = costs.loc[e,"R_Level_2"]
                r3_marg.loc[e,2022] = costs.loc[e,"R_Level_3"]
        print("Initialization done...")
        for t in time:
            if t == year:
                for e in other_elements:
                    if e != element:
                        f_ext_marg.loc[e, t] = min(always_value_capacity(capacity, e, t), f_ext_supply.loc[e, t])
                        deficit_extraction_cap_marg.loc[e, t] = max(0, f_ext_supply.loc[e, t] - always_value_capacity(capacity, e, t))
                    elif e == element:
                        f_ext_marg.loc[e, t] = min(always_value_capacity(capacity, e, t), f_ext_supply.loc[e, t]*(1+fract))
                        print("f_ext_marg = ",f_ext_marg.loc[e, t])
                        print("f_ext = ",f_ext.loc[e, t])
                        deficit_extraction_cap_marg.loc[e, t] = max(0, f_ext_supply.loc[e, t]*(1+fract) - always_value_capacity(capacity, e, t))
                for b in pure_byproducts:
                    if b != element:
                        upper_bound_f_ext_marg.loc[b,t] = min(always_value_capacity(capacity, b, t),sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts))

                        f_ext_marg.loc[b, t] = min(upper_bound_f_ext_marg.loc[b,t], f_ext_supply.loc[b, t])

                        deficit_extraction_cap_marg.loc[b,t] = max(0,f_ext_supply.loc[b,t]-upper_bound_f_ext_marg.loc[b,t])
                        surplus_extraction_cap_marg.loc[b,t] = max(0,sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts)-f_ext_supply.loc[b, t])
                    elif b == element:
                        print("b = ",b)
                        upper_bound_f_ext_marg.loc[b,t] = min(always_value_capacity(capacity, b, t),sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts))
                        f_ext_marg.loc[b, t] = min(upper_bound_f_ext_marg.loc[b,t], f_ext_supply.loc[b, t]*(1+fract))
                        deficit_extraction_cap_marg.loc[b,t] = max(0,f_ext_supply.loc[b,t]*(1+fract)-upper_bound_f_ext_marg.loc[b,t])
                        surplus_extraction_cap_marg.loc[b,t] = max(0,sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts)-f_ext_supply.loc[b, t]*(1+fract))
            elif t != year:
                for e in other_elements:
                    f_ext_marg.loc[e, t] = min(always_value_capacity(capacity, e, t), f_ext_supply.loc[e, t])
                    deficit_extraction_cap_marg.loc[e, t] = max(0, f_ext_supply.loc[e, t] - always_value_capacity(capacity, e, t))
                for b in pure_byproducts:
                    upper_bound_f_ext_marg.loc[b,t] = min(always_value_capacity(capacity, b, t),sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts))
                    f_ext_marg.loc[b, t] = min(upper_bound_f_ext_marg.loc[b,t], f_ext_supply.loc[b, t])
                    deficit_extraction_cap_marg.loc[b,t] = max(0,f_ext_supply.loc[b,t]-upper_bound_f_ext_marg.loc[b,t])
                    surplus_extraction_cap_marg.loc[b,t] = max(0,sum(f_ext_marg.loc[h, t]*BtH_value(mat_BtH, h, b) for h in hosts)-f_ext_supply.loc[b, t])
        print("Now calculating mining energy cost...")
        
        
        
        #print(df_cost_ref)
        (df_cost_marg, total_marg,new_gap_reserves_marg) = mining_cost_calc(element,f_ext_marg,r0_marg,r1_marg,r2_marg,r3_marg,f0_marg,f1_marg,f2_marg,f3_marg,mining_cost_marg,gap_reserves_marg,ts)
        #print(df_cost_marg)
        print("Element ",element)
        print((fract*f_ext_supply.loc[element,year]))
        print("**** RESEDA ****")
        CF_RESEDA.loc[element,year] = (deficit_extraction_cap_marg.loc[element].sum()-deficit_extraction_cap.loc[element].sum())/(fract*f_ext_supply.loc[element,year])
        print(CF_RESEDA.loc[element,year])
        print("**** ACP ****")
        CF_ACP.loc[element,year] = (df_cost_marg.loc[element].sum()-df_cost_ref.loc[element].sum())/(fract*f_ext_supply.loc[element,year])
        print(CF_ACP.loc[element,year])

Scenario =  DLS_IEA_NZ
Initialization done...
f_ext_marg =  147996.63607790924
f_ext =  146531.3228494151
Now calculating mining energy cost...
Element  Mg
1465.313228494151
**** RESEDA ****
0.0
**** ACP ****
1.1602530000007942
Initialization done...
f_ext_marg =  158897.2444749362
f_ext =  157324.0044306299
Now calculating mining energy cost...
Element  Mg
1573.240044306299
**** RESEDA ****
0.0
**** ACP ****
1.1602530000005877
Initialization done...
f_ext_marg =  170610.54284180937
f_ext =  168921.3295463459
Now calculating mining energy cost...
Element  Mg
1689.213295463459
**** RESEDA ****
0.0
**** ACP ****
1.1602529999991802
Initialization done...
f_ext_marg =  183182.17838205094
f_ext =  181368.4934475752
Now calculating mining energy cost...
Element  Mg
1813.684934475752
**** RESEDA ****
0.0
**** ACP ****
1.1602529999996405
Initialization done...
f_ext_marg =  196757.5300548602
f_ext =  194809.4356978814
Now calculating mining energy cost...
Element  Mg
1948.094356978814
**** RES